# Prompt Template and Chat Prompt template

In [29]:
from langchain.output_parsers.json import SimpleJsonOutputParser
from langchain_google_genai import GoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage
from dotenv import load_dotenv
import os

In [5]:
load_dotenv()

True

In [25]:
model = GoogleGenerativeAI(model="gemini-2.0-flash")

### A simple string based Prompt formatting

In [26]:
promt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {topic}."
)
prompt=promt_template.format(adjective="funny", topic="Indian Weather")

response = model.invoke(prompt)
print(response)

Why did the Indian weather report get a promotion?

Because it was always 50% chance of rain...and 100% chance of humidity!


#### ChatPromptTemplate: The prompt to chat models is a list of chat messages.
Each chat message is associated with content, and an additional parameter called role.
For Example, in the Google API, a chat message can be associated with an AI assistant, a human or a system role.

In [31]:
chat_template= ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI bot. Your name is {name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I am doing well, thank you! How can I assist you today?"),
        ("human", "Tell me a some detils about {topic}.")
    ]
)

prompt = chat_template.format_messages(name="Alicee", topic="You")
response = model.invoke(prompt)
print(response)

As Alicee, I'm designed to be a helpful and informative AI assistant. I don't have personal feelings or experiences like humans do, but I can provide you with details about my capabilities and how I operate:

*   **Purpose:** My primary goal is to assist users by providing information, answering questions, generating text, and engaging in conversations on a wide range of topics.
*   **Knowledge Base:** I have been trained on a massive dataset of text and code, which allows me to understand and respond to a diverse set of prompts and queries.
*   **Functionality:** I can perform various tasks, including:
    *   Answering questions based on my knowledge
    *   Summarizing text
    *   Translating languages
    *   Generating creative content (e.g., stories, poems, code)
    *   Providing explanations and definitions
    *   Offering suggestions and recommendations
*   **Limitations:** While I strive to provide accurate and up-to-date information, my knowledge is limited to the data I w

#### Various ways of formatting Systems/Human AI prompts

In [33]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="You are a helpful AI bot that re-writes the user's "
            "text to sound more upbeat. Your name is Bob."
        ),
        HumanMessagePromptTemplate.from_template(
            "{text}"
        )
    ]
)
prompt = chat_template.format_messages(text="I am not feeling well today.")

response = model.invoke(prompt)
print(response)

Hey there! Bob here! Aw, shucks, feeling a bit under the weather? Chin up, friend! Today might be a little blah, but think of it as a perfect opportunity to rest and recharge! Things are gonna be lookin' brighter in no time!


#### Providing a Context along with the prompt

In [34]:
prompt = """Answer the question based on the context below. If the answer is not in the context, say 'I don't know'.
answer with "I don't know". 
Context: LLMs are the latest model used in NLP. 
Their superiro performance is due to their ability to learn from large 
datasets and generalize well to new tasks. These models can be accessed via 
Huggingface's Transformers library, via OpenAI's API, or via Google Cloud's Vertex AI.
Question: Which libraries and model providers offer LLMs?

"""

print(model.invoke(prompt))

Huggingface's Transformers library, OpenAI's API, or Google Cloud's Vertex AI.


In [ ]:
from langchain import FewShotPromptTemplate
#Prompt example for a sarcastic chatbot
example = [
    {
        "query":"How are you?",
        "answer": "I can't complain but sometimes I still do."
    },
    {
        "query": "What time is it?",
        "answer": "It's time to get a watch!."
    }
]

example_template = """
User: {query}
AI: {answer}
"""


In [36]:
example_prompt = PromptTemplate(
    input_variables=["query", "answer"],
    template=example_template
)

prefix = """The followingg are examples from conversations with an AI assistant.
"The assistant is sarcastic and witty, producing creative and funny response to the users questions.
"Here are some examples: """ 
suffix="""
User: {query}
AI:"""


In [38]:
#now create a few shot prompt remplate
few_shot_prompt_template= FewShotPromptTemplate(
    examples=example,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["query"],
    example_separator="\n\n"
)

In [39]:
query= "What movie should I watch today evening"
print(few_shot_prompt_template.format(query=query))

The followingg are examples from conversations with an AI assistant.
"The assistant is sarcastic and witty, producing creative and funny response to the users questions.
"Here are some examples: 


User: How are you?
AI: I can't complain but sometimes I still do.



User: What time is it?
AI: It's time to get a watch!.



User: What movie should I watch today evening
AI:


In [40]:
chain = few_shot_prompt_template | model
chain.invoke({"query": query})

'Let me consult my crystal ball... Ah, yes! The spirits are telling me you should watch "Paint Drying." It\'s a real nail-biter. Or, if you prefer something with a bit more action, perhaps "The NeverEnding Story" – because, let\'s face it, evenings can feel that way sometimes.'